# DSMarket — Task 5: Pipeline and API for operational forecasting

<div style="background-color:#2D8C4E;border-left:6px solid #2D8C4E;padding:14px;border-radius:10px">

**Notebook objective**

Demonstrate how to move the project's final solution into a minimal operational form that can be consumed by other systems.

This notebook does not retrain the model.  
Its purpose is to show:

- how to prepare input data,
- how to load the final recommended model,
- how to apply operational calibration,
- and how to expose an order recommendation through a simple API.

</div>

<a id="indice"></a>

## Table of Contents

1. [What this notebook demonstrates](#que-demuestra)
2. [Setup and required inputs](#setup)
3. [Artifact loading and configuration](#artefactos)
4. [Minimum preparation pipeline](#pipeline)
5. [Model application and calibration](#modelo-calibracion)
6. [API design](#api)
7. [Inference demonstration](#inferencia)
8. [Executive conclusion](#cierre)

<a id="que-demuestra"></a>

## What this notebook demonstrates

This notebook represents the transition from notebook-based analysis to a minimum operational solution.

The logic is as follows:

1. load the necessary data and artifacts,
2. prepare input variables with a reproducible pipeline,
3. apply the final recommended model,
4. correct the prediction using the calibration validated in Task 3,
5. and show how that output could be exposed through an API.

The operational solution used as the reference in this notebook is:

- **technical winning model:** E2
- **recommended operational solution:** calibrated E2

[⬆ Back to table of contents](#indice)

In [3]:
# =============================================================================
# Imports y configuración global
# =============================================================================

import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

COLORS = {
    "primary":   "#2D8C4E",
    "secondary": "#F4A261",
    "accent":    "#E76F51",
    "neutral":   "#8ECAE6",
    "dark":      "#264653",
}

CALIBRATION_FACTOR = 1.3657

print(f"✅ Configuración cargada | SEED={SEED}")
print(f"✅ Factor de calibración operativo: {CALIBRATION_FACTOR}")


✅ Configuración cargada | SEED=42
✅ Factor de calibración operativo: 1.3657


<a id="setup"></a>

## Setup and required inputs

This notebook needs to distinguish between two types of inputs:

### Base project data
- `daily_calendar_with_events.csv`
- `item_prices.csv`
- `item_sales.csv`

### Final model artifacts
- `model_E2.txt`
- E2 calibration factor

### Scope decision

In this task, the operational solution is based on **calibrated E2**.  
Therefore:

- **loading product clusters is not mandatory**
- **model retraining is not necessary**
- **running the full forecasting process again is not required**

[⬆ Back to table of contents](#indice)

In [6]:
# =============================================================================
# Rutas portables del proyecto
# =============================================================================

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR_DEFAULT = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = OUTPUTS_DIR / "forecasting_models"

for path in [INTERIM_DIR, PROCESSED_DIR, FIGURES_DIR, OUTPUTS_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

REQUIRED_RAW_FILES = {
    "daily_calendar_with_events.csv",
    "item_prices.csv",
    "item_sales.csv",
}

def has_required_files(folder: Path) -> bool:
    return folder.is_dir() and REQUIRED_RAW_FILES.issubset({p.name for p in folder.iterdir() if p.is_file()})

def build_candidate_raw_dirs() -> list[Path]:
    candidates = []

    env_raw = os.getenv("DSMARKET_RAW_DIR")
    if env_raw:
        candidates.append(Path(env_raw))

    candidates.extend([
        PROJECT_ROOT / "data" / "raw",
        PROJECT_ROOT / "raw",
        PROJECT_ROOT.parent / "data" / "raw",
        Path("/content/DSMarket/data/raw"),
        Path("/content/drive/MyDrive/TFM MASTER/TFM/Mateo/data/raw"),
        Path("/kaggle/input/dsmarket"),
        Path("/kaggle/input/datasets/mateopascual/dsmarket"),
    ])

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in kaggle_input.iterdir():
            if path.is_dir():
                candidates.append(path)
                candidates.extend([sub for sub in path.iterdir() if sub.is_dir()])

    unique_candidates = []
    seen = set()

    for path in candidates:
        path = Path(path)
        if str(path) not in seen:
            unique_candidates.append(path)
            seen.add(str(path))

    return unique_candidates

def resolve_raw_dir() -> Path:
    candidates = build_candidate_raw_dirs()

    for candidate in candidates:
        if has_required_files(candidate):
            return candidate
        if has_required_files(candidate / "data" / "raw"):
            return candidate / "data" / "raw"
        if has_required_files(candidate / "raw"):
            return candidate / "raw"

    searched = "\n - ".join(str(p) for p in candidates[:20])
    raise FileNotFoundError(
        "No se encontró una carpeta válida con los archivos requeridos.\n"
        "Archivos esperados:\n"
        f" - {chr(10).join(sorted(REQUIRED_RAW_FILES))}\n\n"
        "Puedes definir manualmente la variable de entorno DSMARKET_RAW_DIR.\n"
        f"Primeras rutas revisadas:\n - {searched}"
    )

RAW_DIR = resolve_raw_dir()

print("✅ RAW_DIR resuelto:", RAW_DIR)
print("✅ MODELS_DIR:", MODELS_DIR)

✅ RAW_DIR resuelto: /kaggle/input/datasets/mateopascual/dsmarket
✅ MODELS_DIR: /kaggle/working/outputs/forecasting_models


In [11]:
# =============================================================================
# Detección de artefactos del modelo final
# =============================================================================

def resolve_model_e2_path() -> Path | None:
    candidates = [
        MODELS_DIR / "model_E2.txt",
        OUTPUTS_DIR / "forecasting_models" / "model_E2.txt",
        PROJECT_ROOT / "models" / "model_E2.txt",
        PROJECT_ROOT / "outputs" / "forecasting_models" / "model_E2.txt",
        RAW_DIR / "model_E2.txt",
    ]

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    return None

MODEL_E2_PATH = resolve_model_e2_path()

if MODEL_E2_PATH is None:
    print("ℹ️ No se encontró model_E2.txt todavía.")
    print("ℹ️ El notebook podrá dejar preparado el pipeline y la API,")
    print("ℹ️ pero la inferencia real con el modelo quedará pendiente hasta cargar ese artefacto.")
else:
    print("✅ Modelo E2 encontrado en:", MODEL_E2_PATH)

✅ Modelo E2 encontrado en: /kaggle/input/datasets/mateopascual/dsmarket/model_E2.txt


In [12]:
# =============================================================================
# Carga de datos base
# =============================================================================

sales_dtypes = {
    "item": "category",
    "store_code": "category",
    "category": "category",
}

prices_dtypes = {
    "item": "category",
    "store_code": "category",
}

df_sales = pd.read_csv(
    RAW_DIR / "item_sales.csv",
    dtype=sales_dtypes,
    low_memory=False
)

df_prices = pd.read_csv(
    RAW_DIR / "item_prices.csv",
    dtype=prices_dtypes,
    low_memory=False
)

df_cal = pd.read_csv(
    RAW_DIR / "daily_calendar_with_events.csv",
    parse_dates=["date"],
    low_memory=False
)

print("✅ Datos base cargados")
print("df_sales :", df_sales.shape)
print("df_prices:", df_prices.shape)
print("df_cal   :", df_cal.shape)

✅ Datos base cargados
df_sales : (30490, 1920)
df_prices: (6965706, 5)
df_cal   : (1913, 5)


In [13]:
# =============================================================================
# Resumen de inputs disponibles
# =============================================================================

inputs_status = pd.DataFrame([
    {"input": "daily_calendar_with_events.csv", "disponible": True},
    {"input": "item_prices.csv",                "disponible": True},
    {"input": "item_sales.csv",                 "disponible": True},
    {"input": "model_E2.txt",                   "disponible": MODEL_E2_PATH is not None},
    {"input": "factor_calibracion",             "disponible": True},
])

display(inputs_status)

,input,disponible
0,daily_calendar_with_events.csv,True
1,item_prices.csv,True
2,item_sales.csv,True
3,model_E2.txt,True
4,factor_calibracion,True


<a id="artefactos"></a>

## Artifact loading and configuration

In this section, I load the final recommended model and define the minimum elements required for inference.

The reference operational logic will be:

- **base model:** E2
- **operational correction:** global calibration
- **final output:** calibrated forecast and order recommendation

[⬆ Back to table of contents](#indice)

In [14]:
# =============================================================================
# Carga del modelo final E2
# =============================================================================

model_E2 = None

if MODEL_E2_PATH is not None:
    model_E2 = lgb.Booster(model_file=str(MODEL_E2_PATH))
    print("✅ model_E2 cargado correctamente")
    print("Ruta:", MODEL_E2_PATH)
else:
    print("ℹ️ No se pudo cargar model_E2 porque no se encontró model_E2.txt")

✅ model_E2 cargado correctamente
Ruta: /kaggle/input/datasets/mateopascual/dsmarket/model_E2.txt


In [15]:
# =============================================================================
# Features operativas del modelo E2
# =============================================================================

FEATURES_E2_API = [
    "lag_7",
    "lag_14",
    "lag_21",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "year",
    "is_weekend",
    "has_event",
]

print("✅ Features operativas definidas")
print(FEATURES_E2_API)

✅ Features operativas definidas
['lag_7', 'lag_14', 'lag_21', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'day_of_week', 'day_of_month', 'week_of_year', 'month', 'year', 'is_weekend', 'has_event']


<a id="pipeline"></a>

## Minimum preparation pipeline

The objective here is not to reconstruct the full history from notebook 3,
but to demonstrate how an operational input for inference would be prepared.

The minimum pipeline logic will be:

1. select a store × product combination,
2. retrieve its recent history,
3. build the necessary features,
4. apply the model,
5. calibrate the output,
6. and convert it into an order recommendation.

[⬆ Back to table of contents](#indice)

In [16]:
# =============================================================================
# Preparación mínima del panel diario
# =============================================================================

day_cols = [c for c in df_sales.columns if re.fullmatch(r"d_\d+", str(c))]
day_cols = sorted(day_cols, key=lambda x: int(str(x).split("_")[1]))

meta_cols = ["item", "store_code", "category"]

df_panel_api = df_sales[meta_cols + day_cols].melt(
    id_vars=meta_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="units"
)

df_cal_slim = df_cal[["d", "date"]].copy()
df_panel_api = df_panel_api.merge(df_cal_slim, on="d", how="left")

df_panel_api["date"] = pd.to_datetime(df_panel_api["date"], errors="coerce")
df_panel_api["units"] = pd.to_numeric(df_panel_api["units"], errors="coerce").fillna(0).astype("float32")

df_panel_api = df_panel_api.sort_values(["store_code", "item", "date"]).reset_index(drop=True)

print("✅ df_panel_api preparado")
print(df_panel_api.shape)
display(df_panel_api.head(5))

✅ df_panel_api preparado
(58327370, 6)


,item,store_code,category,d,units,date
0,ACCESORIES_1_001,BOS_1,ACCESORIES,d_1,0.0000,2011-01-29
1,ACCESORIES_1_001,BOS_1,ACCESORIES,d_2,0.0000,2011-01-30
2,ACCESORIES_1_001,BOS_1,ACCESORIES,d_3,0.0000,2011-01-31
3,ACCESORIES_1_001,BOS_1,ACCESORIES,d_4,0.0000,2011-02-01
4,ACCESORIES_1_001,BOS_1,ACCESORIES,d_5,0.0000,2011-02-02


In [17]:
# =============================================================================
# Construcción mínima de features para inferencia
# =============================================================================

df_panel_api["day_of_week"] = df_panel_api["date"].dt.dayofweek.astype("int8")
df_panel_api["day_of_month"] = df_panel_api["date"].dt.day.astype("int8")
df_panel_api["week_of_year"] = df_panel_api["date"].dt.isocalendar().week.astype("int16")
df_panel_api["month"] = df_panel_api["date"].dt.month.astype("int8")
df_panel_api["year"] = df_panel_api["date"].dt.year.astype("int16")
df_panel_api["is_weekend"] = (df_panel_api["day_of_week"] >= 5).astype("int8")

for lag in [7, 14, 21, 28]:
    df_panel_api[f"lag_{lag}"] = (
        df_panel_api.groupby(["store_code", "item"], observed=True)["units"]
        .shift(lag)
        .astype("float32")
    )

for window in [7, 28]:
    df_panel_api[f"rolling_mean_{window}"] = (
        df_panel_api.groupby(["store_code", "item"], observed=True)["units"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        .astype("float32")
    )

if "event" in df_cal.columns:
    df_event = df_cal[["date", "event"]].copy()
    df_event["has_event"] = df_event["event"].notna().astype("int8")
else:
    df_event = df_cal[["date"]].copy()
    df_event["has_event"] = 0

df_panel_api = df_panel_api.drop(columns=["has_event"], errors="ignore")
df_panel_api = df_panel_api.merge(df_event[["date", "has_event"]], on="date", how="left")
df_panel_api["has_event"] = df_panel_api["has_event"].fillna(0).astype("int8")

print("✅ Features mínimas para API construidas")
display(df_panel_api[["store_code", "item", "date"] + FEATURES_E2_API].head(5))

✅ Features mínimas para API construidas


,store_code,item,date,lag_7,lag_14,lag_21,lag_28,rolling_mean_7,rolling_mean_28,day_of_week,day_of_month,week_of_year,month,year,is_weekend,has_event
0,BOS_1,ACCESORIES_1_001,2011-01-29,NaN,NaN,NaN,NaN,NaN,NaN,5,29,4,1,2011,1,0
1,BOS_1,ACCESORIES_1_001,2011-01-30,NaN,NaN,NaN,NaN,0.0000,0.0000,6,30,4,1,2011,1,0
2,BOS_1,ACCESORIES_1_001,2011-01-31,NaN,NaN,NaN,NaN,0.0000,0.0000,0,31,5,1,2011,0,0
3,BOS_1,ACCESORIES_1_001,2011-02-01,NaN,NaN,NaN,NaN,0.0000,0.0000,1,1,5,2,2011,0,0
4,BOS_1,ACCESORIES_1_001,2011-02-02,NaN,NaN,NaN,NaN,0.0000,0.0000,2,2,5,2,2011,0,0


In [18]:
# =============================================================================
# Snapshot más reciente por tienda × producto
# =============================================================================

df_latest_snapshot = (
    df_panel_api
    .sort_values(["store_code", "item", "date"])
    .groupby(["store_code", "item"], observed=True, as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

print("✅ Último snapshot creado")
print(df_latest_snapshot.shape)
display(df_latest_snapshot[["store_code", "item", "category", "date"] + FEATURES_E2_API].head(10))

✅ Último snapshot creado
(30490, 19)


,store_code,item,category,date,lag_7,lag_14,lag_21,lag_28,rolling_mean_7,rolling_mean_28,day_of_week,day_of_month,week_of_year,month,year,is_weekend,has_event
0,BOS_1,ACCESORIES_1_001,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,0.0000,0.2857,0.2857,6,24,16,4,2016,1,0
1,BOS_1,ACCESORIES_1_002,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,0.0000,0.0000,0.0357,6,24,16,4,2016,1,0
2,BOS_1,ACCESORIES_1_003,ACCESORIES,2016-04-24,0.0000,1.0000,0.0000,0.0000,0.0000,0.2143,6,24,16,4,2016,1,0
3,BOS_1,ACCESORIES_1_004,ACCESORIES,2016-04-24,2.0000,2.0000,3.0000,2.0000,0.7143,0.8214,6,24,16,4,2016,1,0
4,BOS_1,ACCESORIES_1_005,ACCESORIES,2016-04-24,2.0000,2.0000,0.0000,0.0000,1.7143,0.8929,6,24,16,4,2016,1,0
5,BOS_1,ACCESORIES_1_006,ACCESORIES,2016-04-24,1.0000,2.0000,0.0000,0.0000,0.2857,0.4643,6,24,16,4,2016,1,0
6,BOS_1,ACCESORIES_1_007,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,2.0000,0.2857,0.3214,6,24,16,4,2016,1,0
7,BOS_1,ACCESORIES_1_008,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,4.0000,1.4286,2.2143,6,24,16,4,2016,1,0
8,BOS_1,ACCESORIES_1_009,ACCESORIES,2016-04-24,0.0000,0.0000,0.0000,0.0000,0.4286,0.3929,6,24,16,4,2016,1,0
9,BOS_1,ACCESORIES_1_010,ACCESORIES,2016-04-24,1.0000,0.0000,1.0000,4.0000,0.4286,0.3929,6,24,16,4,2016,1,0


In [20]:
# =============================================================================
# Validación de columnas necesarias para inferencia
# =============================================================================

missing_cols = [c for c in FEATURES_E2_API if c not in df_latest_snapshot.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas necesarias para inferencia: {missing_cols}")

null_summary = df_latest_snapshot[FEATURES_E2_API].isna().sum()

print("✅ Validación de inferencia completada")
print("\nNaNs por feature:")
display(null_summary.to_frame("n_nulls").T)

✅ Validación de inferencia completada

NaNs por feature:


,lag_7,lag_14,lag_21,lag_28,rolling_mean_7,rolling_mean_28,day_of_week,day_of_month,week_of_year,month,year,is_weekend,has_event
n_nulls,0,0,0,0,0,0,0,0,0,0,0,0,0


In [22]:
# =============================================================================
# Inferencia con model_E2
# =============================================================================

if model_E2 is None:
    print("ℹ️ No se puede hacer inferencia real porque no está cargado model_E2.txt")
    df_inference = df_latest_snapshot.copy()
    df_inference["pred_E2_raw"] = np.nan
else:
    X_api = df_latest_snapshot[FEATURES_E2_API].values.astype("float32")

    df_inference = df_latest_snapshot.copy()
    df_inference["pred_E2_raw"] = np.clip(model_E2.predict(X_api), 0, None).astype("float32")

    print("✅ Inferencia E2 completada")
    display(
        df_inference[
            ["store_code", "item", "category", "date", "pred_E2_raw"]
        ].head(10)
    )

✅ Inferencia E2 completada


,store_code,item,category,date,pred_E2_raw
0,BOS_1,ACCESORIES_1_001,ACCESORIES,2016-04-24,0.0000
1,BOS_1,ACCESORIES_1_002,ACCESORIES,2016-04-24,0.0000
2,BOS_1,ACCESORIES_1_003,ACCESORIES,2016-04-24,0.0000
3,BOS_1,ACCESORIES_1_004,ACCESORIES,2016-04-24,1.0025
4,BOS_1,ACCESORIES_1_005,ACCESORIES,2016-04-24,1.0049
5,BOS_1,ACCESORIES_1_006,ACCESORIES,2016-04-24,0.0001
6,BOS_1,ACCESORIES_1_007,ACCESORIES,2016-04-24,0.0000
7,BOS_1,ACCESORIES_1_008,ACCESORIES,2016-04-24,1.7621
8,BOS_1,ACCESORIES_1_009,ACCESORIES,2016-04-24,0.0000
9,BOS_1,ACCESORIES_1_010,ACCESORIES,2016-04-24,0.0080


<a id="modelo-calibracion"></a>

## Model application and calibration

In this section, I apply the final model to a recent snapshot of each store × product combination.

The operational logic is:

1. generate a base prediction with **E2**,
2. correct it with the calibration factor validated in Task 3,
3. and prepare an output that can already be consumed as an operational recommendation.

[⬆ Back to table of contents](#indice)

In [23]:
# =============================================================================
# Aplicación de calibración operativa
# =============================================================================

df_inference = df_inference.copy()

df_inference["pred_E2_calibrated"] = (
    df_inference["pred_E2_raw"] * CALIBRATION_FACTOR
).astype("float32")

print("✅ Calibración aplicada")
display(
    df_inference[
        ["store_code", "item", "pred_E2_raw", "pred_E2_calibrated"]
    ].head(10)
)

✅ Calibración aplicada


,store_code,item,pred_E2_raw,pred_E2_calibrated
0,BOS_1,ACCESORIES_1_001,0.0000,0.0000
1,BOS_1,ACCESORIES_1_002,0.0000,0.0000
2,BOS_1,ACCESORIES_1_003,0.0000,0.0000
3,BOS_1,ACCESORIES_1_004,1.0025,1.3691
4,BOS_1,ACCESORIES_1_005,1.0049,1.3724
5,BOS_1,ACCESORIES_1_006,0.0001,0.0002
6,BOS_1,ACCESORIES_1_007,0.0000,0.0000
7,BOS_1,ACCESORIES_1_008,1.7621,2.4065
8,BOS_1,ACCESORIES_1_009,0.0000,0.0000
9,BOS_1,ACCESORIES_1_010,0.0080,0.0109


In [24]:
# =============================================================================
# Regla mínima de recomendación de pedido
# =============================================================================

# Demo simple: suponer stock disponible fijo y safety stock proporcional
DEFAULT_STOCK_ON_HAND = 20.0
SAFETY_STOCK_FACTOR = 0.25

df_inference["stock_on_hand_demo"] = DEFAULT_STOCK_ON_HAND
df_inference["safety_stock_demo"] = (
    df_inference["pred_E2_calibrated"] * SAFETY_STOCK_FACTOR
).astype("float32")

df_inference["recommended_order_demo"] = np.maximum(
    0,
    df_inference["pred_E2_calibrated"] + df_inference["safety_stock_demo"] - df_inference["stock_on_hand_demo"]
).astype("float32")

print("✅ Recomendación operativa demo calculada")
display(
    df_inference[
        [
            "store_code",
            "item",
            "pred_E2_calibrated",
            "stock_on_hand_demo",
            "safety_stock_demo",
            "recommended_order_demo"
        ]
    ].head(10)
)

✅ Recomendación operativa demo calculada


,store_code,item,pred_E2_calibrated,stock_on_hand_demo,safety_stock_demo,recommended_order_demo
0,BOS_1,ACCESORIES_1_001,0.0000,20.0000,0.0000,0.0000
1,BOS_1,ACCESORIES_1_002,0.0000,20.0000,0.0000,0.0000
2,BOS_1,ACCESORIES_1_003,0.0000,20.0000,0.0000,0.0000
3,BOS_1,ACCESORIES_1_004,1.3691,20.0000,0.3423,0.0000
4,BOS_1,ACCESORIES_1_005,1.3724,20.0000,0.3431,0.0000
5,BOS_1,ACCESORIES_1_006,0.0002,20.0000,0.0000,0.0000
6,BOS_1,ACCESORIES_1_007,0.0000,20.0000,0.0000,0.0000
7,BOS_1,ACCESORIES_1_008,2.4065,20.0000,0.6016,0.0000
8,BOS_1,ACCESORIES_1_009,0.0000,20.0000,0.0000,0.0000
9,BOS_1,ACCESORIES_1_010,0.0109,20.0000,0.0027,0.0000


<a id="api"></a>

## API design

The objective of this part is not to deploy a real server inside the notebook,
but to show what structure an API capable of returning an order recommendation
based on the project's final model would have.

[⬆ Back to table of contents](#indice)

### Example request

```json
{
  "store_id": "NYC_3",
  "item_id": "PROD_00142",
  "horizon_days": 7,
  "service_level": 0.95,
  "stock_on_hand": 20
}

In [27]:
# =============================================================================
# Selección de un caso demo para respuesta API
# =============================================================================

demo_row = df_inference.iloc[0].copy()

demo_request = {
    "store_id": str(demo_row["store_code"]),
    "item_id": str(demo_row["item"]),
    "horizon_days": 7,
    "service_level": 0.95,
    "stock_on_hand": float(demo_row["stock_on_hand_demo"]),
}

print("✅ Request demo preparado")
print(json.dumps(demo_request, indent=2))

✅ Request demo preparado
{
  "store_id": "BOS_1",
  "item_id": "ACCESORIES_1_001",
  "horizon_days": 7,
  "service_level": 0.95,
  "stock_on_hand": 20.0
}


In [28]:
# =============================================================================
# Construcción de respuesta tipo API
# =============================================================================

demo_response = {
    "store_id": str(demo_row["store_code"]),
    "item_id": str(demo_row["item"]),
    "forecast_units_1d_raw": round(float(demo_row["pred_E2_raw"]), 2) if pd.notna(demo_row["pred_E2_raw"]) else None,
    "forecast_units_1d_calibrated": round(float(demo_row["pred_E2_calibrated"]), 2) if pd.notna(demo_row["pred_E2_calibrated"]) else None,
    "safety_stock": round(float(demo_row["safety_stock_demo"]), 2),
    "recommended_order": round(float(demo_row["recommended_order_demo"]), 2),
    "model_version": "E2_calibrated_v1",
    "service_level": 0.95,
    "confidence": "medium"
}

print("✅ Response demo construida")
print(json.dumps(demo_response, indent=2))

✅ Response demo construida
{
  "store_id": "BOS_1",
  "item_id": "ACCESORIES_1_001",
  "forecast_units_1d_raw": 0.0,
  "forecast_units_1d_calibrated": 0.0,
  "safety_stock": 0.0,
  "recommended_order": 0.0,
  "model_version": "E2_calibrated_v1",
  "service_level": 0.95,
  "confidence": "medium"
}


### Example response

```json
{
  "store_id": "NYC_3",
  "item_id": "PROD_00142",
  "forecast_units_1d_raw": 112.4,
  "forecast_units_1d_calibrated": 153.5,
  "safety_stock": 38.4,
  "recommended_order": 171.9,
  "model_version": "E2_calibrated_v1",
  "service_level": 0.95,
  "confidence": "medium"
}


---

### Cell 27 — Important methodological note

```markdown id="43s6q0"
📌 **Conclusion / Decision**

This demo is not intended to represent a complete production system,
but to demonstrate how the project's final solution can be translated into minimum operational inference.

The relevant sequence is now clear:

- feature preparation,
- application of the E2 model,
- prediction calibration,
- and construction of an API-consumable recommendation.

[⬆ Back to table of contents](#indice)

<a id="cierre"></a>

## Executive conclusion

In this final section, I summarize what this notebook actually demonstrates
and what role it plays within the DSMarket project.

[⬆ Back to table of contents](#indice)

### 8.1 What this task demonstrates

Task 5 does not introduce a new model or a new forecasting experiment.

Its contribution is to demonstrate that the project's final solution
can be moved into a minimum operational form that can be consumed by other systems.

Specifically, this notebook shows that it is possible to:

- prepare inference inputs reproducibly,
- load the final recommended model,
- apply the calibration validated in Task 3,
- and build a structured response compatible with an API.

The operational solution used as the reference is:

- **base technical model:** E2
- **recommended operational solution:** calibrated E2

[⬆ Back to table of contents](#indice)

### 8.2 Real scope of the proposal

This notebook should be interpreted as a **proof of operationalization**,
not as a complete MLOps platform.

What is demonstrated is the minimum logic required to move from analysis to service:

1. **preparation pipeline**
2. **model loading**
3. **calibration application**
4. **operational recommendation calculation**
5. **API request/response structure**

What is not fully implemented here is:

- real production orchestration,
- API authentication and security,
- cloud deployment,
- automatic real-time monitoring,
- nor complete inventory and available stock management.

This is consistent with the scope of the Master's Final Project:
to show a credible technical solution connected to business,
without overstating the actual level of implementation.

[⬆ Back to table of contents](#indice)

### 8.3 Value for business and technology

From a business perspective, this notebook shows that the prediction can already be converted into an interpretable operational recommendation.

From a technology perspective, it shows that the solution can be exposed in a structured way through an API,
which facilitates its future integration with:

- ERP,
- operations tools,
- dashboards,
- or automatic replenishment processes.

In other words, Task 5 connects the project's analytical result
with a minimum architecture for real use.

[⬆ Back to table of contents](#indice)

## Executive memorandum — Response to Task 5

<div style="background-color:#2D8C4E;border-left:6px solid #264653;padding:14px;border-radius:10px">

**Subject:** Pipeline and API to operationalize DSMarket's forecasting solution

The conclusion of this task is that the project's final solution
can already be translated into a minimum operational form that can be consumed by other systems.

The proposed architecture is based on four elements:

- reproducible feature preparation,
- loading of the final recommended model,
- application of operational calibration,
- and exposure of the result through a simple API.

This allows the prediction to stop living only in a notebook
and become an order recommendation that can be integrated with internal business processes.

The solution presented is not intended to be a complete production platform,
but it does demonstrate a realistic transition from analysis to service.

As a result, the recommendation is to use this architecture as the technical basis
for the pilot proposed in Task 4
and later evolve it with monitoring, periodic recalibration and controlled deployment.

</div>

[⬆ Back to table of contents](#indice)

---

*Document prepared by Nicole Chen, Senior Data Scientist — DSMarket*  
*March 2025*